# Explore master flat stock

Inspect housing sources and prototype merges here. The merge example uses **synthetic records**, not actual housing data. Adapt it after inspecting your files and deciding the row grain and join keys.


In [1]:
import pandas as pd

In [ ]:
from pathlib import Path

# Works when the kernel starts at the repository root or inside notebooks/.
start = Path.cwd().resolve()
repo_root = next(
    (p for p in (start, *start.parents)
     if (p / "packages" / "homerun_housing" / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Start the kernel inside the repository or set repo_root explicitly.")

transaction_path = (
    repo_root / "data" / "raw" / "transactions"
    / "ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv"
)
transactions = pd.read_csv(transaction_path, dtype={"block": "string"})
transactions["month"] = pd.to_datetime(transactions["month"], format="%Y-%m", errors="raise")
if transactions.empty or transactions["month"].isna().any():
    raise ValueError("Expected nonempty transactions with a month on every row.")
transactions = transactions.sort_values("month", ascending=False, kind="stable").reset_index(drop=True)
print(f"{len(transactions):,} transactions")
print(f"Date range: {transactions['month'].min():%Y-%m} to {transactions['month'].max():%Y-%m}")
transactions.head(10)


## Explore recent resale transactions

These are historical completed transactions, not active listings. Recent records can form a **simulated candidate pool** for a prototype; their prices are transaction prices, and they do not establish current availability. The loaded file starts in 2017, not 2016.

Explore volume before choosing a holdout window. All windows below use calendar months relative to the latest month in this file, rather than today's date. The latest month may be incomplete or affected by reporting delays. Excluding it does not guarantee the other months are complete. No source files are changed or exported here.


In [ ]:
# Include zero-count months so gaps remain visible.
month_index = pd.period_range(
    transactions["month"].min().to_period("M"),
    transactions["month"].max().to_period("M"),
    freq="M",
)
monthly_counts = (
    transactions.groupby(transactions["month"].dt.to_period("M"))
    .size().reindex(month_index, fill_value=0)
    .rename_axis("month").rename("transaction_count").to_frame()
)
monthly_counts.tail(18).sort_index(ascending=False)


In [ ]:
# Compare the newest month with an endpoint one month earlier.
latest_month = transactions["month"].max().to_period("M")
transaction_months = transactions["month"].dt.to_period("M")
window_rows = []
for include_latest in (True, False):
    end_month = latest_month if include_latest else latest_month - 1
    for months in (1, 2, 3, 6, 12):
        start_month = end_month - (months - 1)
        mask = transaction_months.between(start_month, end_month)
        window_rows.append({
            "includes_latest_month": include_latest,
            "months": months,
            "start_month": str(start_month),
            "end_month": str(end_month),
            "transactions": int(mask.sum()),
            "towns": transactions.loc[mask, "town"].nunique(),
            "flat_types": transactions.loc[mask, "flat_type"].nunique(),
            "share_of_all_transactions_pct": round(mask.mean() * 100, 2),
        })
window_summary = pd.DataFrame(window_rows)
window_summary


### Inspect a candidate window

Change `RECENT_MONTHS` and `INCLUDE_LATEST_MONTH` to explore coverage. The default examines three months ending before the latest observed month. Town and flat-type counts help assess whether common user filters leave enough candidates; an overall count alone cannot establish sufficiency.


In [ ]:
RECENT_MONTHS = 3
INCLUDE_LATEST_MONTH = False
if not isinstance(RECENT_MONTHS, int) or isinstance(RECENT_MONTHS, bool) or RECENT_MONTHS < 1:
    raise ValueError("RECENT_MONTHS must be a positive integer.")
end_month = latest_month if INCLUDE_LATEST_MONTH else latest_month - 1
start_month = end_month - (RECENT_MONTHS - 1)
recent_mask = transaction_months.between(start_month, end_month)
recent_transactions = transactions.loc[recent_mask].copy()
# If later used as a time holdout, historical price references must predate the window.
history_before_window = transactions.loc[transaction_months < start_month].copy()
print(f"Candidate window: {start_month} through {end_month}")
print(f"Simulated candidates: {len(recent_transactions):,}")
print(f"Earlier historical transactions: {len(history_before_window):,}")
print(f"Later records outside this window: {(transaction_months > end_month).sum():,}")
recent_transactions.groupby("town").size().rename("transactions").sort_values(ascending=False).to_frame()


In [ ]:
# Candidate counts after combining town and flat-type filters.
pd.crosstab(recent_transactions["town"], recent_transactions["flat_type"], margins=True)


In [ ]:
from pathlib import Path
import pandas as pd

# Set an absolute Path here only if the kernel starts outside this repository.
REPO_ROOT = None
if REPO_ROOT is None:
    start = Path.cwd().resolve()
    REPO_ROOT = next(
        (path for path in (start, *start.parents)
         if (path / "packages" / "homerun_housing" / "pyproject.toml").is_file()
         and (path / "data").is_dir()),
        None,
    )
if REPO_ROOT is None:
    raise RuntimeError("Set REPO_ROOT to the homerun_data_eng repository path.")
REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
DATA_ROOT = REPO_ROOT / "data"
RAW_DIR = DATA_ROOT / "raw" / "housing_master"
INTERIM_DIR = DATA_ROOT / "interim" / "housing_master"
PROCESSED_DIR = DATA_ROOT / "processed" / "housing_master"
print(f"Repository: {REPO_ROOT}")


## Discover source files

This lists filenames without loading source records. Put original housing files in `data/raw/housing_master/`.


In [ ]:
source_files = sorted(path.relative_to(RAW_DIR).as_posix() for path in RAW_DIR.rglob("*") if path.is_file() and path.name != ".gitkeep")
print("\n".join(source_files) if source_files else "No source files yet.")


## Inspect a real CSV when available

Uncomment and adjust the path below. Initially reading columns as strings preserves postal codes and source identifiers. Convert counts, coordinates and dates explicitly after inspecting their formats. Review outputs before sharing or committing the notebook.


In [ ]:
# source = pd.read_csv(RAW_DIR / "your_source.csv", dtype="string")
# source.info()
# source.head()


## Prototype a merge with synthetic data

This example keeps the block master at one row per block. Dwelling counts remain in a separate table at one row per block and flat type. Replace these examples with your real sources once their schemas are known.


In [ ]:
blocks_demo = pd.DataFrame({
    "block_id": ["demo_001", "demo_002"],
    "postal_code": ["012345", "654321"],
    "year_completed": [1990, 2000],
})
locations_demo = pd.DataFrame({
    "block_id": ["demo_001", "demo_002"],
    "latitude": [1.30, 1.35],
    "longitude": [103.80, 103.85],
})
dwelling_counts_demo = pd.DataFrame({
    "block_id": ["demo_001", "demo_001", "demo_002"],
    "flat_type": ["3 ROOM", "4 ROOM", "4 ROOM"],
    "dwelling_units": [40, 60, 80],
})


In [ ]:
# Check keys before merging. Null keys can match each other in pandas.
for frame in (blocks_demo, locations_demo):
    assert frame["block_id"].notna().all(), "Missing block ID"
    assert not frame["block_id"].duplicated().any(), "Duplicate block ID"

master_demo = blocks_demo.merge(
    locations_demo,
    on="block_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)
print("Location match results:")
print(master_demo["_merge"].value_counts())
assert master_demo["_merge"].eq("both").all(), "Some blocks lack locations"
master_demo = master_demo.drop(columns="_merge")
master_demo


In [ ]:
assert dwelling_counts_demo[["block_id", "flat_type"]].notna().all().all()
assert not dwelling_counts_demo.duplicated(["block_id", "flat_type"]).any()
assert dwelling_counts_demo["dwelling_units"].ge(0).all()
assert dwelling_counts_demo["block_id"].isin(master_demo["block_id"]).all()

# This enriched table has one row per block and flat type.
flat_stock_demo = dwelling_counts_demo.merge(
    master_demo, on="block_id", how="left", validate="many_to_one"
)
flat_stock_demo


## Export experimental results explicitly

The cell below does not write by default. Enable it only if you want the synthetic example written to `data/interim/housing_master/`. Replace the data and output name for real experiments. Promote validated real outputs to a versioned snapshot in `data/processed/housing_master/`.


In [ ]:
WRITE_DEMO_OUTPUT = False
if WRITE_DEMO_OUTPUT:
    INTERIM_DIR.mkdir(parents=True, exist_ok=True)
    output_path = INTERIM_DIR / "synthetic_flat_stock_demo.csv"
    if output_path.exists():
        raise FileExistsError(f"Choose a new output filename: {output_path}")
    flat_stock_demo.to_csv(output_path, index=False)
    print(f"Saved {output_path}")
